In [0]:
import requests
import json
from datetime import datetime

GITHUB_TOKEN = "YOUR_GITHUB_PAT (Personal Access Token)"

headers = {
    "Authorization": f"Bearer {GITHUB_TOKEN}"
}

In [0]:
repos = [
    {"owner": "anthropics", "repo": "claude-plugins-official"},
    {"owner": "colbymchenry", "repo": "codegraph"},
    {"owner": "tinyhumansai", "repo": "openhuman"}
]

today = datetime.today().strftime('%Y-%m-%d')
BASE_PATH = f"/Volumes/workspace/default/github_data/raw"

In [0]:
paths = {
    "repos": f"{BASE_PATH}/repos/{today}/",
    "contributors": f"{BASE_PATH}/contributors/{today}/",
    "commits": f"{BASE_PATH}/commits/{today}/",
    "prs": f"{BASE_PATH}/pull_requests/{today}/"
}

for p in paths.values():
    dbutils.fs.mkdirs(p)

In [0]:
def fetch_paginated_data(url, headers, max_pages=5):
    all_data = []
    for page in range(1, max_pages + 1):
        paginated_url = f"{url}?per_page=100&page={page}"
        response = requests.get(paginated_url, headers=headers)
        if response.status_code != 200:
            print("Error:", response.json())
            break
        data = response.json()
        if not data:
            break
        all_data.extend(data)
    return all_data

In [0]:
repo_data = []
for r in repos:
    url = f"https://api.github.com/repos/{r['owner']}/{r['repo']}"
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        data = response.json()
        data["ingestion_date"] = today
        repo_data.append(data)
print("Repos fetched:", len(repo_data))

Repos fetched: 3


In [0]:
contributors_data = []

for r in repos:
    url = f"https://api.github.com/repos/{r['owner']}/{r['repo']}/contributors"
    data = fetch_paginated_data(url, headers)
    for d in data:
        d["repo"] = r["repo"]
        d["owner"] = r["owner"]
        d["ingestion_date"] = today
    contributors_data.extend(data)
print("Contributors fetched:", len(contributors_data))

Contributors fetched: 144


In [0]:
commits_data = []

for r in repos:
    url = f"https://api.github.com/repos/{r['owner']}/{r['repo']}/commits"
    data = fetch_paginated_data(url, headers, max_pages=5)  # limit pages
    for d in data:
        d["repo"] = r["repo"]
        d["owner"] = r["owner"]
        d["ingestion_date"] = today
    commits_data.extend(data)
print("Commits fetched:", len(commits_data))

Commits fetched: 1212


In [0]:
len(commits_data)

1212

In [0]:
prs_data = []

for r in repos:
    url = f"https://api.github.com/repos/{r['owner']}/{r['repo']}/pulls?state=all"
    data = fetch_paginated_data(url, headers, max_pages=5)
    for d in data:
        d["repo"] = r["repo"]
        d["owner"] = r["owner"]
        d["ingestion_date"] = today
    prs_data.extend(data)
print("PRs fetched:", len(prs_data))

PRs fetched: 159


In [0]:
len(prs_data)

159

In [0]:
with open(f"{paths['repos']}repo_metadata.json", "w") as f:
    json.dump(repo_data, f)

with open(f"{paths['contributors']}contributors.json", "w") as f:
    json.dump(contributors_data, f)

with open(f"{paths['commits']}commits.json", "w") as f:
    json.dump(commits_data, f)

with open(f"{paths['prs']}pull_requests.json", "w") as f:
    json.dump(prs_data, f)